In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

2026-01-14 15:41:28.764239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768405289.051587      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768405289.131704      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768405289.844932      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768405289.844986      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768405289.844990      55 computation_placer.cc:177] computation placer alr

In [2]:
def get_age_group(age):
    if age <= 20:
        return 0   # 0-20
    elif age <= 30:
        return 1   # 21-30
    elif age <= 60:
        return 2   # 31-60
    else:
        return 3   # 60+

In [3]:
IMG_SIZE = 64
images = []
gender_labels = []
age_labels = []

dataset_path = "/kaggle/input/utkface-new/UTKFace" 

for file in os.listdir(dataset_path):
    try:
        age, gender, _, _ = file.split("_")
        age = int(age)
        gender = int(gender)

        img = cv2.imread(os.path.join(dataset_path, file))
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        images.append(img)
        gender_labels.append(gender)
        age_labels.append(get_age_group(age))
    except:
        pass


In [4]:
images[0].shape

(64, 64, 3)

In [5]:
X = np.array(images) / 255.0
y_gender = np.array(gender_labels)
y_age = np.array(age_labels)

In [6]:
X_train, X_test, y_gender_train, y_gender_test = train_test_split(
    X, y_gender, test_size=0.2, random_state=42
)

_, _, y_age_train, y_age_test = train_test_split(
    X, y_age, test_size=0.2, random_state=42
)

In [7]:
gender_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

gender_model.compile(
    optimizer=Adam(0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gender_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-01-14 15:46:20.977217: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,281 (6.20 MB)

 Trainable params: 1,625,281 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
gender_model.fit(
    X_train,
    y_gender_train,
    validation_data=(X_test, y_gender_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 59s 97ms/step - accuracy: 0.6705 - loss: 0.5989 - val_accuracy: 0.8340 - val_loss: 0.3933
Epoch 2/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 82s 97ms/step - accuracy: 0.8299 - loss: 0.3860 - val_accuracy: 0.8555 - val_loss: 0.3337
Epoch 3/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 97ms/step - accuracy: 0.8571 - loss: 0.3337 - val_accuracy: 0.8741 - val_loss: 0.2986
Epoch 4/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 59s 99ms/step - accuracy: 0.8724 - loss: 0.2983 - val_accuracy: 0.8770 - val_loss: 0.2815
Epoch 5/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 98ms/step - accuracy: 0.8778 - loss: 0.2886 - val_accuracy: 0.8756 - val_loss: 0.2769
Epoch 6/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 97ms/step - accuracy: 0.8862 - loss: 0.2678 - val_accuracy: 0.8846 - val_loss: 0.2631
Epoch 7/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 96ms/step - accuracy: 0.8890 - loss: 0.2625 - val_accuracy: 0.8857 - val_loss: 0.2577
Epoch 8/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 96ms/step - accuracy: 0.8923 - loss: 0.2548 - 

In [9]:
gender_model.fit(
    X_train,
    y_gender_train,
    validation_data=(X_test, y_gender_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 98ms/step - accuracy: 0.9063 - loss: 0.2252 - val_accuracy: 0.8897 - val_loss: 0.2515
Epoch 2/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 96ms/step - accuracy: 0.9098 - loss: 0.2222 - val_accuracy: 0.8924 - val_loss: 0.2441
Epoch 3/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 96ms/step - accuracy: 0.9109 - loss: 0.2149 - val_accuracy: 0.8950 - val_loss: 0.2405
Epoch 4/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 59s 99ms/step - accuracy: 0.9197 - loss: 0.2033 - val_accuracy: 0.8943 - val_loss: 0.2466
Epoch 5/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.9173 - loss: 0.2013 - val_accuracy: 0.8971 - val_loss: 0.2377
Epoch 6/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 59s 100ms/step - accuracy: 0.9193 - loss: 0.1975 - val_accuracy: 0.8973 - val_loss: 0.2374
Epoch 7/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.9234 - loss: 0.1921 - val_accuracy: 0.8983 - val_loss: 0.2325
Epoch 8/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.9255 - loss: 0.1861 -

In [14]:
gender_model.save("gender_model.h5")

In [10]:
y_age_train_cat = to_categorical(y_age_train, num_classes=4)
y_age_test_cat = to_categorical(y_age_test, num_classes=4)

In [11]:
age_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

age_model.compile(
    optimizer=Adam(0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

age_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,668 (6.20 MB)

 Trainable params: 1,625,668 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
age_model.fit(
    X_train,
    y_age_train_cat,
    validation_data=(X_test, y_age_test_cat),
    epochs=15,
    batch_size=32
)

Epoch 1/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 96ms/step - accuracy: 0.4527 - loss: 1.1992 - val_accuracy: 0.5969 - val_loss: 0.9095
Epoch 2/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 97ms/step - accuracy: 0.6037 - loss: 0.9309 - val_accuracy: 0.6558 - val_loss: 0.8217
Epoch 3/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 57s 96ms/step - accuracy: 0.6367 - loss: 0.8502 - val_accuracy: 0.6743 - val_loss: 0.7810
Epoch 4/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 98ms/step - accuracy: 0.6657 - loss: 0.8022 - val_accuracy: 0.6788 - val_loss: 0.7460
Epoch 5/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.6697 - loss: 0.7827 - val_accuracy: 0.6874 - val_loss: 0.7336
Epoch 6/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 59s 100ms/step - accuracy: 0.6789 - loss: 0.7566 - val_accuracy: 0.6845 - val_loss: 0.7222
Epoch 7/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 97ms/step - accuracy: 0.6956 - loss: 0.7317 - val_accuracy: 0.7041 - val_loss: 0.7081
Epoch 8/15
593/593 ━━━━━━━━━━━━━━━━━━━━ 58s 98ms/step - accuracy: 0.6965 - loss: 0.7180 -

In [13]:
age_model.save("age_model.h5")